In [1]:
import tensorflow as tf
import numpy as np

#1. Tensorflow를 통한 MNIST 다운로드
(x_train,y_train), (x_test,y_test)= tf.keras.datasets.mnist.load_data()
#2. 데이터 형태 변환 및 정규화
x_train=x_train.reshape(-1,1,28,28).astype(np.float32)
x_test=x_test.reshape(-1,1,28,28).astype(np.float32)
x_train/=255.0
x_test/=255.0

#3 정답지를 one-hot encoding으로 바꾸기
def one_hot_encoding(y,num_classes=10):
    return np.eye(num_classes)[y]

y_train_encoding=one_hot_encoding(y_train)
y_test_encoding=one_hot_encoding(y_test)


In [2]:
import numpy as np

class ReLU:
    def __init__(self):
        self.mask=None

    def forward(self,x):
        self.mask=(x<=0)
        out=x.copy()
        out[self.mask]=0
        return out
    def backward(self,dout):
        dout[self.mask]=0
        dx=dout
        return dx


In [3]:
class FullyConnected:
    def __init__(self,W,b):
        self.W=W
        self.b=b

        self.x=None #순전파 시의 입력값을 저장할 변수
        self.dW=None
        self.db=None

    def forward(self,x):
        self.x=x
        out=np.dot(self.x,self.W)+self.b
        return out
    def backward(self,dout):
        dx=np.dot(dout,self.W.T)
        self.dW=np.dot(self.x.T,dout)
        self.db=np.sum(dout,axis=0)

        return dx

In [4]:
def softmax(x):
    if x.ndim==2:
        x=x.T
        x=x-np.max(x,axis=0)
        y= np.exp(x)/np.sum(np.exp(x),axis=0)  
        return y.T
    x=x-np.max(x)
    return np.exp(x)/np.sum(np.exp(x))

def cross_entropy_error(y,t):
    if y.ndim==1:
        t=t.reshape(1,t.size)
        y=y.reshape(1,y.size)
    if t.size==y.size:
        t=t.argmax(axis=1)
    batch_size=y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size),t]+1e-7))/batch_size


In [5]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss=None
        self.y=None
        self.t=None 
    def forward(self,x,t):
        self.t=t
        self.y=softmax(x)
        self.loss=cross_entropy_error(self.y,self.t)
        return self.loss
    def backward(self,dout=1):
        batch_size=self.t.shape[0]
        dx=(self.y-self.t)/batch_size
        return dx

In [6]:
def im2col(input_data,filter_h,filter_w,stride=1,pad=0):
    N,C,H,W=input_data.shape 
    out_h=(H+2*pad-filter_h)//stride+1
    out_w=(W+2*pad-filter_w)//stride+1
    img=np.pad(input_data,[(0,0),(0,0),(pad,pad),(pad,pad)],'constant')
    col=np.zeros((N,C,filter_h,filter_w,out_h,out_w))

    for y in range(filter_h):
        y_max=y+stride*out_h
        for x in range(filter_w):
            x_max=x+stride*out_w
            col[:,:,y,x,:,:]=img[:,:,y:y_max:stride,x:x_max:stride]
    col=col.transpose(0,4,5,1,2,3).reshape(N*out_h*out_w,-1)
    return col

In [7]:
def col2im(col,input_shape,filter_h,filter_w,stride=1,pad=0):
    N,C,H,W=input_shape
    out_h=(H+2*pad-filter_h)//stride+1
    out_w=(W+2*pad-filter_w)//stride+1
    col=col.reshape(N,out_h,out_w,C,filter_h,filter_w).transpose(0,3,4,5,1,2)
    img=np.zeros((N,C,H+2*pad+stride-1,W+2*pad+stride-1))

    for y in range(filter_h):
        y_max=y+stride*out_h
        for x in range(filter_w):
            x_max=x+stride*out_w
            img[:,:,y:y_max:stride,x:x_max:stride]+=col[:,:,y,x,:,:]
    return img[:,:,pad:H+pad,pad:W+pad]

In [18]:
class Convolution:
    def __init__(self,W,b,stride=1,pad=0):
        self.W=W
        self.b=b
        self.stride=stride
        self.pad=pad

        self.x=None
        self.col=None
        self.col_W=None
        self.dW=None 
        self.db=None

    def forward(self,x):
        FN,C,FH,FW=self.W.shape
        N,C,H,W=x.shape

        out_h=int(1+(H+2*self.pad-FH)/self.stride)
        out_w=int(1+(W+2*self.pad-FW)/self.stride)

        col=im2col(x,FH,FW,self.stride,self.pad)

        col_W=self.W.reshape(FN,-1).T 
        out=np.dot(col,col_W)+self.b
        out=out.reshape(N,out_h,out_w,-1).transpose(0,3,1,2)

        self.x=x
        self.col=col
        self.col_W=col_W

        return out
    def backward(self,dout):
        FN,C,FH,FW=self.W.shape 
        dout=dout.transpose(0,2,3,1).reshape(-1,FN)

        self.db=np.sum(dout,axis=0)
        self.dW=np.dot(self.col.T,dout)

        self.dW=self.dW.transpose(1,0).reshape(FN,C,FH,FW)
        dcol=np.dot(dout,self.col_W.T)
        dx=col2im(dcol,self.x.shape,FH,FW,self.stride,self.pad)
        return dx

In [14]:
class Pooling:
    def __init__(self,pool_h,pool_w,stride=1,pad=0):
        self.pool_h=pool_h
        self.pool_w=pool_w
        self.stride=stride
        self.pad=pad

        self.x=None
        self.arg_max=None
    def forward(self,x):
        N,C,H,W=x.shape
        out_h=int(1+(H-self.pool_h)/self.stride)
        out_w=int(1+(W-self.pool_w)/self.stride)

        col=im2col(x,self.pool_h,self.pool_w,self.stride,self.pad)
        col=col.reshape(-1,self.pool_h*self.pool_w)
        arg_max=np.argmax(col,axis=1)
        out=np.max(col,axis=1)
        out=out.reshape(N,out_h,out_w,C).transpose(0,3,1,2)
        self.x=x
        self.arg_max=arg_max
        return out
    def backward(self,dout):
        dout=dout.transpose(0,2,3,1)
        pool_size=self.pool_h*self.pool_w
        dmax=np.zeros((dout.size,pool_size))
        dmax[np.arange(self.arg_max.size),self.arg_max.flatten()]=dout.flatten()
        dmax=dmax.reshape(dout.shape+(pool_size,))
        dcol=dmax.reshape(dmax.shape[0]*dmax.shape[1]*dmax.shape[2],-1)
        dx=col2im(dcol,self.x.shape,self.pool_h,self.pool_w,self.stride,self.pad)
        return dx

In [10]:
from collections import OrderedDict

class SimpleConvNet:
    def __init__(self):
        self.params={}
        self.params['W1']=np.random.randn(30,1,5,5)*0.01
        self.params['b1']=np.zeros(30)
        self.params['W2']=np.random.rand(4320,100)*0.01
        self.params['b2']=np.zeros(100)
        self.params['W3']=np.random.rand(100,10)*0.01
        self.params['b3']=np.zeros(10)

        self.layers=OrderedDict()
        self.layers['Conv1']=Convolution(self.params['W1'],self.params['b1'],stride=1,pad=0)
        self.layers['Relu1']=ReLU()
        self.layers['Pool1']=Pooling(pool_h=2,pool_w=2,stride=2)
        self.layers['Affine1']=FullyConnected(self.params['W2'],self.params['b2'])
        self.layers['Relu2']=ReLU()
        self.layers['Affine2']=FullyConnected(self.params['W3'],self.params['b3'])
        self.last_layer=SoftmaxWithLoss()

    def predict(self,x):
        for key,layer in self.layers.items():
            if key=='Affine1':
                x=x.reshape(x.shape[0],-1)
            x=layer.forward(x)
        return x
    def loss(self,x,t):
        y=self.predict(x)
        return self.last_layer.forward(y,t)
    def gradient(self,x,t):
        self.loss(x,t)

        dout=1
        dout=self.last_layer.backward(dout)

        layers=list(self.layers.values())
        layers.reverse()

        for layer in layers: 
            if isinstance(layer,Pooling):
                dout=dout.reshape(dout.shape[0],30,12,12)
            dout=layer.backward(dout)
        grads={}
        grads['W1'],grads['b1']=self.layers['Conv1'].dW,self.layers['Conv1'].db
        grads['W2'],grads['b2']=self.layers['Affine1'].dW,self.layers['Affine1'].db
        grads['W3'],grads['b3']=self.layers['Affine2'].dW,self.layers['Affine2'].db

        return grads


In [19]:
network=SimpleConvNet()

iters_num=1000
train_size=x_train.shape[0]
batch_size=100
learning_rate=0.1
train_loss_list=[]
print("===학습 시작===")
for i in range(iters_num):
    batch_mask=np.random.choice(train_size,batch_size)
    x_batch=x_train[batch_mask]
    t_batch=y_train_encoding[batch_mask]
    grads=network.gradient(x_batch,t_batch)
    for key in('W1','b1','W2','b2','W3','b3'):
        network.params[key]-=learning_rate*grads[key]
    loss=network.loss(x_batch,t_batch)
    train_loss_list.append(loss)
    if i% 100==0:
        print(f"{i}번째 반복 - 현재 Loss: {loss:.4f}")
print("===학습완료===")


===학습 시작===
0번째 반복 - 현재 Loss: 2.2964
100번째 반복 - 현재 Loss: 2.1217
200번째 반복 - 현재 Loss: 1.9506
300번째 반복 - 현재 Loss: 0.7046
400번째 반복 - 현재 Loss: 0.3442
500번째 반복 - 현재 Loss: 0.1799
600번째 반복 - 현재 Loss: 0.0761
700번째 반복 - 현재 Loss: 0.1258
800번째 반복 - 현재 Loss: 0.0902
900번째 반복 - 현재 Loss: 0.0680
===학습완료===


In [20]:
def calculate_accuracy(network, x, t, batch_size=100):
    """
    메모리 초과를 막기 위해 테스트 데이터도 배치 단위로 쪼개서 정확도를 계산합니다.
    """
    acc = 0.0
    # 전체 데이터를 batch_size로 나눈 횟수만큼 반복
    for i in range(int(x.shape[0] / batch_size)):
        tx = x[i * batch_size : (i + 1) * batch_size]
        tt = t[i * batch_size : (i + 1) * batch_size]
        
        # 1. 모델을 통한 예측 (Softmax 통과 전의 점수)
        y = network.predict(tx)
        
        # 2. 확률(점수)이 가장 높은 인덱스를 예측된 숫자로 선택
        y_pred = np.argmax(y, axis=1)
        t_true = np.argmax(tt, axis=1)
        
        # 3. 정답과 일치하는 개수 누적
        acc += np.sum(y_pred == t_true)
        
    # 전체 데이터 개수로 나누어 최종 정확도 산출
    return acc / x.shape[0]

# ==========================================
# 🚀 최종 테스트 데이터 정확도 평가 실행
# ==========================================
print("=== 테스트 데이터 정확도 평가 중 ===")
test_acc = calculate_accuracy(network, x_test, y_test_encoding)
print(f"최종 테스트 정확도 (Accuracy): {test_acc * 100:.2f}%")

=== 테스트 데이터 정확도 평가 중 ===
최종 테스트 정확도 (Accuracy): 96.05%
